# 04 · Análise Exploratória das Taxas

**Objetivo:** explorar as taxas e identificar perfis de maior/menor risco.

**Entradas:** `data/processed/rates/level_*.parquet`.  
**Saídas:** figuras em `outputs/figures/`, tabelas em `outputs/tables/`.

**Limitações:** associações descritivas, não causais.

In [ ]:
# Preâmbulo: torna o pacote src importável a partir do notebook
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src.config import load_config, anos_validos
cfg = load_config()
print('Raiz:', cfg['root'], '| modo sintético:', cfg['synthetic_mode'])

In [ ]:
from src import rates, viz
tables, meta = rates.load_level_tables(cfg['abs']['rates'])
fig_dir, tab_dir = cfg['abs']['figures'], cfg['abs']['tables']

# Dois recortes: nível RICO (muitas dimensões, células pequenas) para as
# marginais ponderadas; nível mais AGREGADO (células populosas) para a
# distribuição e o ranking, onde queremos taxas estáveis por célula.
lvl_rico, lvl_agg = 'cbo4_cnae2_tempo_tam_uf', 'cbo2_cnae2_tempo_uf'
df_rico = tables[lvl_rico].copy()
df_rico['taxa_sjc'] = df_rico['k_involuntario_sjc'] / df_rico['n']
df_agg = tables[lvl_agg].copy()
df_agg['taxa_sjc'] = df_agg['k_involuntario_sjc'] / df_agg['n']
exp_min = cfg['suavizacao']['exposicao_minima']
print(f'rico={len(df_rico):,} células (n médio {df_rico.n.mean():.0f}) | '
      f'agg={len(df_agg):,} células (n médio {df_agg.n.mean():.0f})')

In [ ]:
# Distribuição das taxas no nível agregado (células com exposição mínima)
df_ok = df_agg[df_agg['n'] >= exp_min]
fig = viz.plot_rate_distribution(df_ok['taxa_sjc'],
       f'Distribuição da taxa anual (dispensa s/ justa causa) — n>={exp_min}')
viz.save_fig(fig, fig_dir / 'dist_taxa_sjc.png'); fig

In [ ]:
# Efeitos marginais por dimensão (média ponderada por exposição no nível rico)
for dim in ['tempo_faixa', 'tamanho_faixa', 'uf']:
    fig = viz.plot_marginal(df_rico, dim, 'taxa_sjc',
            f'Risco involuntário médio por {dim}')
    viz.save_fig(fig, fig_dir / f'marginal_{dim}.png')
    display(fig)

In [ ]:
# Ranking de perfis de MAIOR risco (com suporte) -> tabela de output.
# Exclui CBO/CNAE não-identificados ('00') e exige exposição mais robusta.
rk = df_ok[(df_ok['cbo2'] != '00') & (df_ok['cnae2'] != '00') & (df_ok['n'] >= 200)]
rank = (rk.sort_values('taxa_sjc', ascending=False)
          .head(25)[['cbo2','cnae2','tempo_faixa','uf','n','taxa_sjc']])
rank.to_csv(tab_dir / 'perfis_maior_risco.csv', index=False)
display(rank)

In [ ]:
# Ranking de perfis de MENOR risco (com suporte) -> tabela de output
rank_min = (df_ok.sort_values('taxa_sjc', ascending=True)
                 .head(25)[['cbo2','cnae2','tempo_faixa','uf','n','taxa_sjc']])
rank_min.to_csv(tab_dir / 'perfis_menor_risco.csv', index=False)
display(rank_min)

In [ ]:
# Cobertura do backoff: exposição por nível
cob = pd.DataFrame([
    {'nivel': name, 'n_celulas': len(t), 'exposicao_total': int(t['n'].sum())}
    for name, t in tables.items()])
cob.to_csv(tab_dir / 'cobertura_niveis.csv', index=False)
display(cob)